# Prompt Complexity Classifier Training (Colab GPU)

**Instructions:**
1. **Runtime → Change runtime type → T4 GPU** (IMPORTANT!)
2. Run all cells in order
3. Download the trained model at the end

Expected time: **5-10 minutes** on T4 GPU

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn torch
print("✅ Dependencies installed!")

## Step 2: Verify GPU

In [ ]:
import torch

print("="*80)
print("GPU CHECK")
print("="*80)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ WARNING: GPU not detected!")
    print("Go to Runtime → Change runtime type → T4 GPU")
print("="*80)

## Step 3: Fetch Datasets from Hugging Face

In [ ]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

# Create directories
CACHE_DIR = Path("data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Fetching datasets...\n")

# 1. WildChat
print("[1/3] Fetching WildChat...")
dataset = load_dataset("allenai/WildChat", split="train", streaming=True)
prompts = []
for i, example in enumerate(dataset):
    if len(prompts) >= 20000:
        break
    if example.get("language") != "English":
        continue
    conversation = example.get("conversation", [])
    if conversation and conversation[0].get("role") == "user":
        prompt_text = conversation[0].get("content", "").strip()
        if 10 <= len(prompt_text) <= 2000:
            prompts.append({"prompt": prompt_text, "source_dataset": "wildchat"})
wildchat_df = pd.DataFrame(prompts)
wildchat_df.to_parquet(CACHE_DIR / "wildchat_sampled.parquet", index=False)
print(f"✅ WildChat: {len(wildchat_df):,} prompts\n")

# 2. GSM8K
print("[2/3] Fetching GSM8K...")
dataset = load_dataset("openai/gsm8k", "main", split="train")
gsm8k_df = pd.DataFrame([{"prompt": ex["question"], "source_dataset": "gsm8k"} for ex in dataset])
gsm8k_df.to_parquet(CACHE_DIR / "gsm8k.parquet", index=False)
print(f"✅ GSM8K: {len(gsm8k_df):,} prompts\n")

# 3. SupraLabs
print("[3/3] Fetching SupraLabs...")
dataset = load_dataset("SupraLabs/Prompt-Routing-Dataset", split="train")
supralabs_df = dataset.to_pandas()
supralabs_df["source_dataset"] = "supralabs"
if "text" in supralabs_df.columns:
    supralabs_df = supralabs_df.rename(columns={"text": "prompt"})
supralabs_df.to_parquet(CACHE_DIR / "supralabs_routing.parquet", index=False)
print(f"✅ SupraLabs: {len(supralabs_df):,} prompts\n")

print(f"✅ Total: {len(wildchat_df) + len(gsm8k_df) + len(supralabs_df):,} prompts")

## Step 4: Apply Complexity Labels

In [ ]:
import re

# Label SupraLabs
def map_supralabs_labels(df):
    def assign_label(row):
        if row['routing_choice'] == 'small model':
            return 'small'
        elif row['routing_choice'] == 'big model':
            return 'medium' if row['complexity_score'] <= 2 else 'large'
        return 'medium'
    df['complexity_label'] = df.apply(assign_label, axis=1)
    return df

# Heuristic labeler
def heuristic_label(prompt, source):
    text_lower = prompt.lower()
    token_count = len(prompt) // 4
    
    if source == 'gsm8k':
        return 'medium' if token_count < 40 else 'large'
    
    reasoning_kw = {'step by step', 'explain why', 'prove', 'calculate', 'solve', 'analyze'}
    math_sym = {'=', '+', '-', '*', '/', '^', '√', '∫', '∑', '\\frac', '\\int'}
    code_markers = {'def ', 'class ', 'import ', '```', 'python', 'function'}
    simple_pat = {'what is', 'who is', 'when was', 'define'}
    
    has_reasoning = any(kw in text_lower for kw in reasoning_kw)
    has_math = any(sym in prompt for sym in math_sym)
    has_code = any(marker in text_lower for marker in code_markers)
    is_simple = any(pat in text_lower[:50] for pat in simple_pat)
    
    if is_simple and token_count < 50:
        return 'small'
    if token_count < 30 and not (has_reasoning or has_math or has_code):
        return 'small'
    if has_reasoning or has_math or '```' in prompt or token_count > 300:
        return 'large'
    if has_code or (100 <= token_count <= 300):
        return 'medium'
    return 'medium'

# Apply labels
supralabs_labeled = map_supralabs_labels(supralabs_df)
wildchat_df['complexity_label'] = wildchat_df['prompt'].apply(lambda x: heuristic_label(x, 'wildchat'))
gsm8k_df['complexity_label'] = gsm8k_df['prompt'].apply(lambda x: heuristic_label(x, 'gsm8k'))

# Combine
full_df = pd.concat([
    supralabs_labeled[['prompt', 'source_dataset', 'complexity_label']],
    wildchat_df[['prompt', 'source_dataset', 'complexity_label']],
    gsm8k_df[['prompt', 'source_dataset', 'complexity_label']]
], ignore_index=True)

# Save
LABELED_DIR = Path("data/labeled")
LABELED_DIR.mkdir(parents=True, exist_ok=True)
full_df.to_parquet(LABELED_DIR / "prompt_complexity_labeled.parquet", index=False)

print(f"✅ Combined dataset: {len(full_df):,} samples")
print(f"\nClass distribution:")
print(full_df['complexity_label'].value_counts())

## Step 5: Train the Classifier (GPU-Optimized)

In [ ]:
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import Dataset

# Load data
df = pd.read_parquet(LABELED_DIR / "prompt_complexity_labeled.parquet")

# Train/val split (stratified)
df['strat_key'] = df['complexity_label'] + "_" + df['source_dataset']
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['strat_key'])
train_df = train_df.drop('strat_key', axis=1)
val_df = val_df.drop('strat_key', axis=1)

print(f"Train: {len(train_df):,} samples")
print(f"Val:   {len(val_df):,} samples\n")

# Label mapping
LABEL2ID = {"small": 0, "medium": 1, "large": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Load tokenizer
MODEL_NAME = "microsoft/MiniLM-L12-H384-uncased"
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenize datasets
def tokenize_dataset(df, tokenizer):
    df['label'] = df['complexity_label'].map(LABEL2ID)
    dataset = Dataset.from_pandas(df[['prompt', 'label']])
    def tokenize_fn(examples):
        return tokenizer(examples['prompt'], truncation=True, max_length=256, padding=False)
    return dataset.map(tokenize_fn, batched=True, remove_columns=['prompt'])

print("Tokenizing datasets...")
train_dataset = tokenize_dataset(train_df, tokenizer)
val_dataset = tokenize_dataset(val_df, tokenizer)
print("✅ Tokenization complete\n")

# Load model
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
print(f"✅ Model loaded on device: {'cuda' if torch.cuda.is_available() else 'cpu'}\n")

# Training arguments (GPU-optimized)
training_args = TrainingArguments(
    output_dir="./checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,  # Increased for GPU
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    fp16=True,  # Mixed precision
    dataloader_pin_memory=True,
    report_to="none",
)

# Compute metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'f1_weighted': f1_score(labels, predictions, average='weighted'),
    }

# Trainer
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train
print("="*80)
print("🚀 STARTING TRAINING")
print("="*80)
train_start = time.time()
trainer.train()
train_time = time.time() - train_start

print(f"\n✅ Training completed in {train_time:.2f}s ({train_time/60:.2f} min)")

## Step 6: Evaluate Model

In [ ]:
# Evaluate
predictions = trainer.predict(val_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

accuracy = accuracy_score(true_labels, pred_labels)
f1_macro = f1_score(true_labels, pred_labels, average='macro')
cm = confusion_matrix(true_labels, pred_labels)

print("="*80)
print("EVALUATION RESULTS")
print("="*80)
print(f"Accuracy:     {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1 (macro):   {f1_macro:.4f}")
print(f"\nConfusion Matrix:")
print(f"           small  medium  large")
for i, label in enumerate(['small', 'medium', 'large']):
    print(f"{label:8s}: " + "  ".join(f"{v:5d}" for v in cm[i]))
print("="*80)

## Step 7: Save and Download Model

In [ ]:
# Save model
MODEL_DIR = Path("prompt_complexity_classifier")
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

# Save training report
with open(MODEL_DIR / "training_report.txt", "w") as f:
    f.write(f"Training Time: {train_time:.2f}s ({train_time/60:.2f} min)\n")
    f.write(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)\n")
    f.write(f"F1 (macro): {f1_macro:.4f}\n")
    f.write(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}\n")
    f.write(f"\nConfusion Matrix:\n")
    f.write(f"           small  medium  large\n")
    for i, label in enumerate(['small', 'medium', 'large']):
        f.write(f"{label:8s}: " + "  ".join(f"{v:5d}" for v in cm[i]) + "\n")

print(f"\n✅ Model saved to {MODEL_DIR}")

# Zip for download
print("\n📦 Zipping model for download...")
!zip -r prompt_complexity_classifier.zip prompt_complexity_classifier/

print("\n" + "="*80)
print("🎉 TRAINING COMPLETE!")
print("="*80)
print("\nNext steps:")
print("1. Download 'prompt_complexity_classifier.zip' from the Files panel (left sidebar)")
print("2. Extract to your local: model/prompt_complexity_classifier/")
print("3. Continue with the remaining steps in your local environment")
print("="*80)